In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import os
import platform
import datetime
import math
import matplotlib.pyplot as plt
import traceback
import random
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

class Config:
    INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS = 13, 128, 2
    BATCH_SIZE, NUM_EPOCHS, LEARNING_RATE = 32, 5, 0.0005  # Changed epochs to 2
    SEQ_LENGTH = 15
    MIN_SPEED, MAX_SPEED = 0.0, 10.0
    MIN_ACC, MAX_ACC = -1.5, 3.0
    DELTA_T = 1/30
    TTC_STAR = 30.0
    MAX_ROWS = 10000
    PHI_SMOOTH_FACTOR = 0.5
    SPACING_SAFE = 15.0
    SPACING_CRITICAL = 2.0

    @staticmethod
    def get_paths():
        if platform.system() == "Windows":
            return {
                'base_dir': r"C:\Users\yliu117\Desktop\IEEE IV",
                'velocity_csv': r'C:\Users\yliu117\Desktop\SVO\ARED\ExpA_leadVel.csv',
                'spacing_csv': r'C:\Users\yliu117\Desktop\SVO\ARED\ExpA_spacing.csv'
            }
        else:
            home = os.path.expanduser("~")
            return {
                'base_dir': os.path.join(home, "IEEE IV"),
                'velocity_csv': os.path.join(home, "SVO", "ARED", "ExpA_leadVel.csv"),
                'spacing_csv': os.path.join(home, "SVO", "ARED", "ExpA_spacing.csv")
            }

class UnifiedIDMCalculator:
    def __init__(self, idm_params, device='cpu'):
        self.params = idm_params
        self.device = device

    def compute(self, current_speed, front_speed, spacing, tensor=False):
        v_0, a_max, b, s_0, T = self.params['v_0'], self.params['a_max'], self.params['b'], self.params['s_0'], self.params['T']
        delta = self.params.get('delta', 4)
        if tensor:
            spacing = torch.clamp(spacing, min=1.0)
            current_speed = torch.clamp(current_speed, min=0.1)
            s_star = s_0 + torch.clamp(
                current_speed * T + (current_speed * (current_speed - front_speed)) /
                (2 * torch.sqrt(torch.tensor(a_max * b, device=self.device))), min=0
            )
            a_free = a_max * (1 - (current_speed / v_0) ** delta)
            a_interaction = -a_max * (s_star / spacing) ** 2
            return torch.clamp(a_free + a_interaction, min=-b * 3, max=a_max * 1.5)
        spacing = max(float(spacing), 1.0)
        current_speed = max(float(current_speed), 0.1)
        front_speed = float(front_speed)
        s_star = s_0 + max(0, current_speed * T + (current_speed * (current_speed - front_speed)) / (2 * math.sqrt(a_max * b)))
        a_free = a_max * (1 - (current_speed / v_0) ** delta)
        a_interaction = -a_max * (s_star / spacing) ** 2
        return max(-b * 3, min(a_free + a_interaction, a_max * 1.5))

class EnhancedFeatureProcessor:
    @staticmethod
    def compute_features(v1, v2, s12, prev_acc=0.0, v1_history=None, v2_history=None):
        try:
            v1_norm, v2_norm, s12_norm = float(v1) / 6.0, float(v2) / 6.0, float(s12) / 30.0
            v3_raw = float(v1)
            v3_acc_raw = (v1_history[-1] - v1_history[-2]) / 0.033 if v1_history is not None and len(v1_history) >= 2 else 0.0
            relative_speed = float(v1 - v2) / 3.0
            v3_acc_norm = v3_acc_raw / 0.6
            v1_acc_norm = (v1_history[-1] - v1_history[-2]) / 0.033 / 0.6 if v1_history is not None and len(v1_history) >= 2 else 0.0
            v1_trend_norm = (v1_history[-1] - v1_history[-3]) / 0.066 / 3.0 if v1_history is not None and len(v1_history) >= 3 else 0.0
            relative_speed_abs = abs(relative_speed)
            ttc_norm = 2.0
            if relative_speed_abs > 0.01 and s12 > 0.1:
                ttc = s12 / (relative_speed * 6.0)
                ttc_norm = min(max(ttc / 5.0, 0.0), 2.0)
                ttc_norm = 1 / (1 + math.exp(-5 * (ttc_norm - 1)))
            urgency = max(0.0, min(1.0, (relative_speed * 2.0 + (1.0/max(s12, 0.1)) * 10.0)))
            relative_speed_trend = 0.0
            if v1_history is not None and v2_history is not None and len(v1_history) >= 2 and len(v2_history) >= 2:
                v1_trend = (v1_history[-1] - v1_history[-2]) / 0.033
                v2_trend = (v2_history[-1] - v2_history[-2]) / 0.033
                relative_speed_trend = (v1_trend - v2_trend) / 3.0
            features = [v1_norm, v2_norm, s12_norm, relative_speed, float(prev_acc) / 0.6, v3_acc_norm, v1_acc_norm, v1_trend_norm, ttc_norm, urgency, relative_speed_trend, v3_raw / 6.0, v3_acc_raw / 0.6]
            return torch.FloatTensor([0.0 if not np.isfinite(feat) else feat for feat in features])
        except Exception as e:
            print(f"Feature computation error: {e}")
            return torch.zeros(13)

    @staticmethod
    def init_history(data, seq_len):
        features, v1_history, v2_history = [], [], []
        for i in range(seq_len):
            idx = seq_len - 1 - i
            v1, v2 = data['speed_3'].iloc[idx], data['speed_4'].iloc[idx]
            v1_history.append(v1)
            v2_history.append(v2)
            features.append(EnhancedFeatureProcessor.compute_features(
                v1, v2, data['spacing_3_4'].iloc[idx], 0.0, v1_history[-10:], v2_history[-10:]
            ))
        return torch.stack(features).float()

    @staticmethod
    def compute_spacing_danger(spacing, safe_threshold=15.0, critical_threshold=1.0):
        if spacing >= safe_threshold:
            return 0.0
        elif spacing <= critical_threshold:
            return 1.0
        else:
            normalized = (safe_threshold - spacing) / (safe_threshold - critical_threshold)
            return 1 / (1 + math.exp(-5 * (normalized - 0.5)))

    @staticmethod
    def compute_dynamic_phi(states, config):
        try:
            safe_spacing = config.SPACING_SAFE
            critical_spacing = config.SPACING_CRITICAL
            s12 = float(states.get('spacing_1_2', safe_spacing))
            s23 = float(states.get('spacing_2_3', safe_spacing))
            s34 = float(states.get('spacing_3_4', safe_spacing))
            D_f = (EnhancedFeatureProcessor.compute_spacing_danger(s12, safe_spacing, critical_spacing) +
                   EnhancedFeatureProcessor.compute_spacing_danger(s23, safe_spacing, critical_spacing) +
                   EnhancedFeatureProcessor.compute_spacing_danger(s34, safe_spacing, critical_spacing)) / 3.0
            s45 = float(states.get('spacing_4_5', safe_spacing))
            s56 = float(states.get('spacing_5_6', safe_spacing))
            D_r = (EnhancedFeatureProcessor.compute_spacing_danger(s45, safe_spacing, critical_spacing) +
                   EnhancedFeatureProcessor.compute_spacing_danger(s56, safe_spacing, critical_spacing)) / 2.0
            if D_f + D_r == 0:
                phi = math.pi / 4
            else:
                phi = math.pi / 2 * (math.tanh(3 * (D_r - D_f)) + 1) / 2
            return max(0.0, min(math.pi / 2, phi))
        except Exception as e:
            print(f"Dynamic phi computation error: {e}")
            return math.pi / 4

    @staticmethod
    def compute_dynamic_phi_with_components(states, config):
        try:
            safe_spacing = config.SPACING_SAFE
            critical_spacing = config.SPACING_CRITICAL
            s12 = float(states.get('spacing_1_2', safe_spacing))
            s23 = float(states.get('spacing_2_3', safe_spacing))
            s34 = float(states.get('spacing_3_4', safe_spacing))
            D_f = (EnhancedFeatureProcessor.compute_spacing_danger(s12, safe_spacing, critical_spacing) +
                   EnhancedFeatureProcessor.compute_spacing_danger(s23, safe_spacing, critical_spacing) +
                   EnhancedFeatureProcessor.compute_spacing_danger(s34, safe_spacing, critical_spacing)) / 3.0
            s45 = float(states.get('spacing_4_5', safe_spacing))
            s56 = float(states.get('spacing_5_6', safe_spacing))
            D_r = (EnhancedFeatureProcessor.compute_spacing_danger(s45, safe_spacing, critical_spacing) +
                   EnhancedFeatureProcessor.compute_spacing_danger(s56, safe_spacing, critical_spacing)) / 2.0
            if D_f + D_r == 0:
                phi = math.pi / 4
            else:
                phi = math.pi / 2 * (math.tanh(3 * (D_r - D_f)) + 1) / 2
            return max(0.0, min(math.pi / 2, phi)), D_f, D_r, np.mean([s12, s23, s34, s45, s56])
        except Exception as e:
            print(f"Dynamic phi computation error: {e}")
            return math.pi / 4, 0.0, 0.0, 15.0

def setup_validation(config=None):
    config = config or Config()
    paths = config.get_paths()
    base_dir = paths['base_dir']
    os.makedirs(base_dir, exist_ok=True)
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    results_dir = os.path.join(base_dir, f"svo_results_{timestamp}")
    os.makedirs(results_dir, exist_ok=True)
    return results_dir

def load_and_validate_data(max_rows=10000, config=None):
    config = config or Config()
    paths = config.get_paths()
    try:
        velocity_data = pd.read_csv(paths['velocity_csv'])
        spacing_data = pd.read_csv(paths['spacing_csv'])
        if not (velocity_data >= 0).all().all() or not (spacing_data > 0).all().all():
            raise ValueError("Invalid speed or spacing values (negative or zero)")
    except FileNotFoundError as e:
        print(f"Data file not found: {e}")
        return None, None
    except ValueError as e:
        print(f"Data validation error: {e}")
        return None, None
    skip_warmup = 500
    skip_ending = 500
    start_idx = skip_warmup
    # Changed to use full MAX_ROWS instead of half
    max_rows = config.MAX_ROWS
    end_idx = min(len(velocity_data) - skip_ending, start_idx + max_rows)
    if end_idx <= start_idx:
        print("Data length too short after halving")
        return None, None
    data_dict = {f'speed_{i}': velocity_data.iloc[start_idx:end_idx, i-1].values for i in range(1, 7)}
    data_dict.update({f'spacing_{i}_{i+1}': spacing_data.iloc[start_idx:end_idx, i-1].values for i in range(1, 6)})
    data = pd.DataFrame(data_dict)
    if data.isnull().any().any() or (data == 0).all().any():
        print("Data contains NaN or all-zero values")
        return None, None
    print(f"Dataset: {len(data)} rows (indices {start_idx}-{end_idx})")
    print("Train Speed std:", {f'speed_{i}': data[f'speed_{i}'].std() for i in range(1, 7)})
    print("Train Spacing std:", {f'spacing_{i}_{i+1}': data[f'spacing_{i}_{i+1}'].std() for i in range(1, 6)})
    split_idx = int(len(data) * 0.8)
    train_data, test_data = data.iloc[:split_idx], data.iloc[split_idx:]
    print(f"Spacing_3_4 stats: mean={test_data['spacing_3_4'].mean():.4f}, min={test_data['spacing_3_4'].min():.4f}, max={test_data['spacing_3_4'].max():.4f}")
    return data, config.DELTA_T

def custom_collate_fn(batch):
    real_states_list, features_list = [], []
    for real_states, features in batch:
        try:
            real_states_list.append(real_states)
            features_list.append(features)
        except Exception as e:
            print(f"Collate error: {e}")
            continue
    if not real_states_list:
        return {}, torch.tensor([])
    batched_states = {key: torch.stack([torch.from_numpy(sample[key]) for sample in real_states_list]) for key in real_states_list[0].keys()}
    return batched_states, torch.stack(features_list)

class NoLeakageSVODataset(Dataset):
    def __init__(self, data, seq_length, model_params):
        self.data = data
        self.seq_length = seq_length
        self.model_params = model_params
        self.valid_indices = list(range(len(data) - seq_length + 1))
        print(f"Dataset initialized: {len(self)} samples")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        sequence_slice = self.data.iloc[actual_idx:actual_idx + self.seq_length]
        real_states = {f'speed_{i}': sequence_slice[f'speed_{i}'].values.astype(np.float32) for i in range(1, 7)}
        real_states.update({f'spacing_{i}_{i+1}': sequence_slice[f'spacing_{i}_{i+1}'].values.astype(np.float32) for i in range(1, 6)})
        features = EnhancedFeatureProcessor.init_history(sequence_slice, self.seq_length)
        return real_states, features

class ImprovedSVOModel(nn.Module):
    def __init__(self, config, model_params):
        super().__init__()
        self.config = config
        self.phi_embedding = nn.Linear(1, 16)
        phi_emb_dim = 17
        self.lstm = nn.LSTM(config.INPUT_SIZE + phi_emb_dim, config.HIDDEN_SIZE, config.NUM_LAYERS, batch_first=True)
        self.dropout = nn.Dropout(0.1)
        self.intent_layer = nn.Linear(config.HIDDEN_SIZE, config.HIDDEN_SIZE)
        self.control_layer = nn.Linear(config.HIDDEN_SIZE + phi_emb_dim, config.HIDDEN_SIZE // 2)
        self.output_layer = nn.Linear(config.HIDDEN_SIZE // 2, 1)
        self._init_weights()
        self.forward_count = 0

    def _init_weights(self):
        for name, param in self.lstm.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data, gain=0.3)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data, gain=0.3)
            elif 'bias' in name:
                param.data.fill_(0)
                n = param.size(0)
                param.data[(n//4):(n//2)].fill_(1)
        for layer in [self.intent_layer, self.control_layer]:
            nn.init.xavier_uniform_(layer.weight, gain=0.3)
            nn.init.zeros_(layer.bias)
        nn.init.normal_(self.output_layer.weight, std=0.2)
        nn.init.normal_(self.output_layer.bias, std=0.1)
        nn.init.xavier_uniform_(self.phi_embedding.weight, gain=0.3)
        nn.init.zeros_(self.phi_embedding.bias)

    def forward(self, x, phi, spacing_34=10.0):
        self.forward_count += 1
        batch_size, seq_len, _ = x.shape
        phi_norm = phi / (math.pi/2)
        phi_input = torch.full((batch_size, 1), phi_norm, device=x.device)
        phi_emb = torch.cat([phi_input, torch.tanh(self.phi_embedding(phi_input))], dim=-1)
        if self.training:
            phi_emb = phi_emb + torch.randn_like(phi_emb) * (0.01 + phi_norm * 0.02)
        phi_exp = phi_emb.unsqueeze(1).expand(-1, seq_len, -1)
        x_phi = torch.cat([x, phi_exp], dim=-1)
        h0 = torch.zeros(self.config.NUM_LAYERS, batch_size, self.config.HIDDEN_SIZE).to(x.device)
        c0 = torch.zeros(self.config.NUM_LAYERS, batch_size, self.config.HIDDEN_SIZE).to(x.device)
        out, _ = self.lstm(x_phi, (h0, c0))
        out = self.dropout(out)
        intent = torch.relu(self.intent_layer(out[:, -1, :]))
        control = torch.relu(self.control_layer(torch.cat([intent, phi_emb], dim=-1)))
        raw_acc = self.output_layer(control)
        if self.training:
            raw_acc = raw_acc + torch.randn_like(raw_acc) * 0.005
        phi_behavior_bias = 0.1 * (phi / (math.pi/2) - 0.5) if self.training else 0.2 * (phi / (math.pi/2) - 0.5)  # Reduced during training
        raw_acc = raw_acc + phi_behavior_bias * 2.5
        acceleration = torch.tanh(raw_acc) * (0.1 + 3.0 * (phi / (math.pi/2)) ** 2.0)
        if self.forward_count % 100 == 0:
            print(f"Phi influence: phi={phi:.4f}, phi_behavior_bias={phi_behavior_bias.item():.4f}, raw_acc={raw_acc.item():.4f}, acceleration={acceleration.item():.4f}, spacing_3_4={spacing_34:.4f}")
        return torch.clamp(acceleration, min=self.config.MIN_ACC, max=self.config.MAX_ACC)

class FastSVOLoss:
    def __init__(self, model_params, dt, device='cpu'):
        self.dt, self.device = dt, device
        self.idm_calc = UnifiedIDMCalculator(model_params['car4']['idm'], device)
        self.car4_v0 = model_params['car4']['idm']['v_0']
        self.u_av_scale, self.u_hv_scale = 0.02, 5.0
        self.trend_weight_max = 50.0

    def compute_realtime_u_av(self, accelerations, dt):
        return sum([0.5 * acc**2 * dt for acc in accelerations]) if len(accelerations) > 0 else 0.0

    def compute_realtime_u_hv_with_idm(self, car2_speeds, initial_car4_speed, initial_spacing_23, dt, phi):
        if len(car2_speeds) < 2:
            return 0.0
        v_0, u_hv_total = self.car4_v0, 0.0
        car4_speed, spacing_23 = max(initial_car4_speed, v_0 * 0.98), initial_spacing_23
        phi_weight = 1.0 - math.sin(phi)
        for car2_speed in car2_speeds:
            car4_acc = self.idm_calc.compute(car4_speed, car2_speed, spacing_23)
            new_car4_speed = np.clip(car4_speed + car4_acc * dt, 0.1, 6.0)
            spacing_23 += (car2_speed - car4_speed) * dt
            u_hv_total += 0.5 * (new_car4_speed - v_0)**2 * dt * phi_weight
            car4_speed = new_car4_speed
        return u_hv_total

    def normalize_svo_components(self, u_av, u_hv):
        return u_av / self.u_av_scale, u_hv / self.u_hv_scale

    def compute_svo_trend_loss(self, phi_values, u_av_values, u_hv_values, epoch):
        if len(phi_values) < 2:
            return torch.tensor(0.0, device=self.device, requires_grad=True)
        phi_values = np.array([float(phi) for phi in phi_values])
        u_av_values = np.array([float(u_av) if not isinstance(u_av, torch.Tensor) else u_av.cpu().item() for u_av in u_av_values])
        u_hv_values = np.array([float(u_hv) if not isinstance(u_hv, torch.Tensor) else u_hv.cpu().item() for u_hv in u_hv_values])
        if not (np.any(u_av_values != 0.0) and np.any(u_hv_values != 0.0)):
            return torch.tensor(0.0, device=self.device, requires_grad=True)
        phi_norm = phi_values / (math.pi / 2)
        u_av_corr = np.corrcoef(phi_norm, u_av_values)[0, 1] if len(phi_values) > 1 else 0.0
        u_hv_corr = np.corrcoef(phi_norm, u_hv_values)[0, 1] if len(phi_values) > 1 else 0.0
        trend_weight = min(epoch / 3.0, 1.0) * self.trend_weight_max if epoch < 5 else self.trend_weight_max
        return trend_weight * (torch.relu(torch.tensor(-u_av_corr, device=self.device)) * 10.0 +
                               torch.relu(torch.tensor(u_hv_corr, device=self.device)) * 10.0)

    def compute_adaptive_weights(self, losses_dict, epoch):
        base_weights = {'follow': 5.0, 'svo': 20.0, 'smoothness': 0.02, 'trend': 15.0}  # Increased follow weight
        if epoch < 3:
            base_weights.update({'follow': 6.0, 'svo': 10.0, 'trend': 4.0})
        elif epoch < 5:
            base_weights.update({'follow': 5.5, 'svo': 15.0, 'trend': 10.0})
        else:
            base_weights.update({'follow': 5.0, 'svo': 20.0, 'trend': 15.0})
        if len(losses_dict) > 1:
            total_loss = sum(losses_dict.values())
            for key in losses_dict:
                loss_ratio = losses_dict[key] / (total_loss + 1e-8)
                base_weights[key] *= 0.5 if loss_ratio > 0.7 else 1.5 if loss_ratio < 0.1 else 1.0
        return base_weights

    def _compute_svo_objective_loss(self, car2_speeds, accelerations, phi, initial_states=None):
        u_av_actual = self.compute_realtime_u_av(accelerations, self.dt)
        initial_car4_speed = self.car4_v0 if initial_states is None or 'car4_speed' not in initial_states else initial_states['car4_speed']
        initial_spacing_23 = 15.0 if initial_states is None or 'spacing_23' not in initial_states else initial_states['spacing_23']
        u_hv_actual = self.compute_realtime_u_hv_with_idm(car2_speeds, initial_car4_speed, initial_spacing_23, self.dt, phi)
        u_av_normalized, u_hv_normalized = self.normalize_svo_components(u_av_actual, u_hv_actual)
        w_av, w_hv = math.cos(phi), math.sin(phi)
        return torch.tensor(w_av * u_av_normalized + w_hv * u_hv_normalized, device=self.device), u_av_normalized, u_hv_normalized

    def compute_enhanced_svo_loss(self, model, states, phi, seq_len, config):
        batch_size = states['speed_1'].shape[0]
        if batch_size == 0:
            return torch.zeros(1, device=self.device, requires_grad=True), []
        phi_list, u_av_list, u_hv_list, individual_losses = [], [], [], []
        total_loss = torch.zeros(1, device=self.device, requires_grad=True)
        for i in range(batch_size):
            sample_data = {f'speed_{j}': states[f'speed_{j}'][i].cpu().numpy() for j in range(1, 7)}
            sample_data.update({f'spacing_{j}_{j+1}': states[f'spacing_{j}_{j+1}'][i].cpu().numpy() for j in range(1, 6)})
            sample_loss, u_av_norm, u_hv_norm = self._process_enhanced_sample_v2(model, sample_data, phi[i], config)
            individual_losses.append(sample_loss)
            phi_list.append(phi[i])
            u_av_list.append(u_av_norm)
            u_hv_list.append(u_hv_norm)
            total_loss = total_loss + sample_loss
        if len(set(phi_list)) > 1:
            total_loss = total_loss + self.compute_svo_trend_loss(phi_list, u_av_list, u_hv_list, getattr(config, 'current_epoch', 0))
        return total_loss / batch_size, []

    def _process_enhanced_sample_v2(self, model, data, phi, config):
        epoch = getattr(config, 'current_epoch', 0)
        speed_1, speed_2, speed_3, speed_4, spacing_23, spacing_34 = [
            torch.tensor(data[f'speed_{i}'], dtype=torch.float32, device=self.device) for i in [1, 2, 3, 4]
        ] + [torch.tensor(data['spacing_2_3'], dtype=torch.float32, device=self.device),
             torch.tensor(data['spacing_3_4'], dtype=torch.float32, device=self.device)]
        seq_len = len(speed_1)
        if seq_len < 3:
            return torch.tensor(0.0, device=self.device, requires_grad=True), 0.0, 0.0
        simulation_start = 2
        loss_calculation_steps = list(range(simulation_start, seq_len, 2)) + ([seq_len - 1] if seq_len - 1 not in list(range(simulation_start, seq_len, 2)) else [])
        total_follow_loss = torch.zeros(1, device=self.device, requires_grad=True)
        total_smoothness_loss = torch.zeros(1, device=self.device, requires_grad=True)
        simulated_speeds = torch.tensor([float(speed_4[0]), float(speed_4[1])], dtype=torch.float32, device=self.device)
        simulated_accelerations = torch.tensor([0.0], dtype=torch.float32, device=self.device)
        v3_history = torch.tensor([float(speed_3[i]) for i in range(min(seq_len, 5))], dtype=torch.float32, device=self.device)
        v4_history = torch.tensor([float(speed_4[i]) for i in range(min(seq_len, 5))], dtype=torch.float32, device=self.device)
        initial_states = {'car4_speed': float(speed_4[0]), 'spacing_23': float(spacing_23[0]), 'spacing_34': float(spacing_34[0])}
        for t in range(simulation_start, seq_len):
            try:
                current_sim_speed = simulated_speeds[-1]
                prev_acceleration = simulated_accelerations[-1]
                if t < len(speed_3):
                    v3_history = torch.cat([v3_history, speed_3[t:t+1]])
                    v4_history = torch.cat([v4_history, torch.tensor([current_sim_speed], device=self.device)])
                if len(v3_history) > 10:
                    v3_history, v4_history = v3_history[-10:], v4_history[-10:]
                enhanced_features = EnhancedFeatureProcessor.compute_features(
                    float(speed_3[t]), float(current_sim_speed), float(spacing_34[t]),
                    float(prev_acceleration), v3_history[-10:].cpu().numpy(), v4_history[-10:].cpu().numpy()
                ).unsqueeze(0).unsqueeze(0).to(self.device)
                phi_tensor = torch.tensor(phi, device=self.device, dtype=torch.float32)
                predicted_acceleration = model(enhanced_features, phi_tensor, spacing_34[t].item()).squeeze()
                if predicted_acceleration.dim() > 0:
                    predicted_acceleration = predicted_acceleration[0]
                new_sim_speed = torch.clamp(
                    current_sim_speed + predicted_acceleration * self.dt,
                    min=config.MIN_SPEED, max=config.MAX_SPEED
                )
                simulated_speeds = torch.cat([simulated_speeds, new_sim_speed.unsqueeze(0)])
                simulated_accelerations = torch.cat([simulated_accelerations, predicted_acceleration.unsqueeze(0)])
                if t in loss_calculation_steps:
                    real_acc = (float(speed_4[t]) - float(speed_4[t-1])) / self.dt if t > 0 else 0.0
                    acc_diff = torch.abs(predicted_acceleration - torch.tensor(real_acc, device=self.device))
                    acc_loss = acc_diff ** 2 * 2.0
                    lookback = min(3, len(simulated_speeds) - 1)
                    trend_loss = 0.0
                    if lookback >= 2:
                        car3_trend = float(speed_3[t]) - float(speed_3[max(0, t-lookback)])
                        car4_trend = new_sim_speed.item() - simulated_speeds[max(0, len(simulated_speeds)-lookback-1)].item()
                        trend_loss = abs(car3_trend - car4_trend) ** 2 * 5.0
                    response_loss = 0.0
                    if len(simulated_accelerations) >= 2 and t < len(speed_3):
                        car3_acc_approx = (float(speed_3[t]) - float(speed_3[t-1])) / self.dt
                        car4_acc = predicted_acceleration.item()
                        if abs(car3_acc_approx) > 0.1:
                            response_loss = (abs(car3_acc_approx) - abs(car4_acc)) ** 2 * 3.0
                    safety_loss = torch.tensor(5.0 if new_sim_speed < config.MIN_SPEED or new_sim_speed > config.MAX_SPEED else 0.0, device=self.device)
                    total_follow_loss = total_follow_loss + acc_loss + trend_loss + response_loss + safety_loss
                    if epoch >= 3 and len(simulated_accelerations) > 1:
                        smoothness_loss = sum((simulated_accelerations[i] - simulated_accelerations[i-1])**2
                                            for i in range(1, len(simulated_accelerations))) * 0.01
                        total_smoothness_loss = total_smoothness_loss + smoothness_loss
            except Exception as sim_error:
                print(f"Error at step {t}: {sim_error}")
                if t > 0 and t < seq_len - 1:
                    simulated_speeds = torch.cat([simulated_speeds, simulated_speeds[-1].unsqueeze(0)])
                    simulated_accelerations = torch.cat([simulated_accelerations, simulated_accelerations[-1].unsqueeze(0)])
                else:
                    break
        svo_objective_loss = torch.tensor(0.0, device=self.device)
        u_av_norm, u_hv_norm = 0.0, 0.0
        if len(simulated_accelerations) > 0 and not (epoch < 2 and len(simulated_accelerations[1:]) < 3):
            svo_objective_loss, u_av_norm, u_hv_norm = self._compute_svo_objective_loss(
                simulated_speeds[2:].detach().cpu().numpy(),
                simulated_accelerations[1:].detach().cpu().numpy(),
                phi, initial_states
            )
        weights = self.compute_adaptive_weights({
            'follow': total_follow_loss.item(),
            'smoothness': total_smoothness_loss.item(),
            'svo': svo_objective_loss.item()
        }, epoch)
        total_loss = (weights['follow'] * total_follow_loss +
                     weights['smoothness'] * total_smoothness_loss +
                     weights['svo'] * svo_objective_loss) / len(loss_calculation_steps)
        return total_loss, u_av_norm, u_hv_norm

def train_enhanced(train_data, model_params, config, device, results_dir):
    model = ImprovedSVOModel(config, model_params).to(device)
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    loss_fn = FastSVOLoss(model_params, config.DELTA_T, device)
    dataset = NoLeakageSVODataset(train_data, config.SEQ_LENGTH, model_params)
    dataloader = DataLoader(dataset, batch_size=config.BATCH_SIZE, shuffle=True, collate_fn=custom_collate_fn)
    history = []
    total_loss = 0.0
    for epoch in range(config.NUM_EPOCHS):
        model.train()
        config.current_epoch = epoch
        for batch_idx, (states, features) in enumerate(dataloader):
            states = {k: v.to(device) for k, v in states.items()}
            features = features.to(device)
            optimizer.zero_grad()
            batch_loss, phi_list = 0.0, []
            batch_size = states['speed_1'].shape[0]
            for i in range(batch_size):
                sample_states = {f'speed_{j}': states[f'speed_{j}'][i, -1].cpu().item() for j in range(1, 7)}
                sample_states.update({f'spacing_{j}_{j+1}': states[f'spacing_{j}_{j+1}'][i, -1].cpu().item() for j in range(1, 6)})
                phi = EnhancedFeatureProcessor.compute_dynamic_phi(sample_states, config)
                phi_list.append(phi)
                sample_data = {f'speed_{j}': states[f'speed_{j}'][i].cpu().numpy() for j in range(1, 7)}
                sample_data.update({f'spacing_{j}_{j+1}': states[f'spacing_{j}_{j+1}'][i].cpu().numpy() for j in range(1, 6)})
                sample_loss, _, _ = loss_fn._process_enhanced_sample_v2(model, sample_data, phi, config)
                batch_loss = batch_loss + sample_loss
            batch_loss = batch_loss / batch_size
            batch_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += batch_loss.item()
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch}, Avg Loss: {avg_loss:.4f}, Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
        scheduler.step(avg_loss)
        history.append(avg_loss)
    print(f"Training completed, Final Avg Loss: {avg_loss:.4f}")
    return model, loss_fn, history

def compute_idm_verification(base_results, model_params, config, compute_spacings=False):
    dt = config.DELTA_T
    real_car1_speeds = np.array(base_results.get('real_car1_speeds', []))
    if real_car1_speeds.size == 0:
        print("real_car1_speeds is empty")
        results = {f'car{i}_speeds': [] for i in range(1, 7)}
        results['accelerations'] = []
        if compute_spacings:
            results['spacings'] = {f'{i}_{i+1}': [] for i in range(1, 6)}
        return results
    states = {}
    for i in range(1, 7):
        key = f'real_car{i}_speeds'
        if key in base_results and len(base_results[key]) > 0:
            states[f'car{i}_speed'] = float(base_results[key][0])
        else:
            states[f'car{i}_speed'] = 1.0
    for i in range(1, 6):
        key = f'{i}_{i+1}'
        if 'real_spacings' in base_results and key in base_results['real_spacings'] and len(base_results['real_spacings'][key]) > 0:
            states[f'spacing_{i}_{i+1}'] = float(base_results['real_spacings'][key][0])
        else:
            states[f'spacing_{i}_{i+1}'] = 6.0
    idm_calcs = {f'car{i}': UnifiedIDMCalculator(model_params[f'car{i}']['idm']) for i in range(2, 7)}
    results = {f'car{i}_speeds': [] for i in range(1, 7)}
    results['accelerations'] = []
    if compute_spacings:
        results['spacings'] = {f'{i}_{i+1}': [] for i in range(1, 6)}
    for i in range(len(real_car1_speeds)):
        try:
            car1_speed = real_car1_speeds[i]
            results['car1_speeds'].append(car1_speed)
            car2_acc = idm_calcs['car2'].compute(states['car2_speed'], car1_speed, states['spacing_1_2'])
            new_car2_speed = np.clip(states['car2_speed'] + car2_acc * dt, config.MIN_SPEED, config.MAX_SPEED)
            results['car2_speeds'].append(new_car2_speed)
            car3_acc = idm_calcs['car3'].compute(states['car3_speed'], new_car2_speed, states['spacing_2_3'])
            new_car3_speed = np.clip(states['car3_speed'] + car3_acc * dt, config.MIN_SPEED, config.MAX_SPEED)
            results['car3_speeds'].append(new_car3_speed)
            car4_acc = idm_calcs['car4'].compute(states['car4_speed'], new_car3_speed, states['spacing_3_4'])
            new_car4_speed = np.clip(states['car4_speed'] + car4_acc * dt, config.MIN_SPEED, config.MAX_SPEED)
            results['car4_speeds'].append(new_car4_speed)
            results['accelerations'].append(car4_acc)
            car5_acc = idm_calcs['car5'].compute(states['car5_speed'], new_car4_speed, states['spacing_4_5'])
            new_car5_speed = np.clip(states['car5_speed'] + car5_acc * dt, config.MIN_SPEED, config.MAX_SPEED)
            results['car5_speeds'].append(new_car5_speed)
            car6_acc = idm_calcs['car6'].compute(states['car6_speed'], new_car5_speed, states['spacing_5_6'])
            new_car6_speed = np.clip(states['car6_speed'] + car6_acc * dt, config.MIN_SPEED, config.MAX_SPEED)
            results['car6_speeds'].append(new_car6_speed)
            speeds = [car1_speed, new_car2_speed, new_car3_speed, new_car4_speed, new_car5_speed, new_car6_speed]
            displacements = [speed * dt for speed in speeds]
            for j in range(1, 6):
                states[f'spacing_{j}_{j+1}'] = max(0.0, states[f'spacing_{j}_{j+1}'] + displacements[j-1] - displacements[j])  # Changed to allow negative, then clip to 0
                if compute_spacings:
                    results['spacings'][f'{j}_{j+1}'].append(max(0.0, states[f'spacing_{j}_{j+1}']))  # Ensure non-negative
            states.update({
                'car1_speed': car1_speed,
                'car2_speed': new_car2_speed,
                'car3_speed': new_car3_speed,
                'car4_speed': new_car4_speed,
                'car5_speed': new_car5_speed,
                'car6_speed': new_car6_speed
            })
        except Exception as e:
            print(f"IDM verification error at step {i}: {e}")
            if compute_spacings:
                for j in range(1, 6):
                    key = f'{j}_{j+1}'
                    last_valid_spacing = results['spacings'][key][-1] if results['spacings'][key] else 10.0
                    results['spacings'][key].append(max(0.0, last_valid_spacing))
    return results

def simulate_enhanced(model, test_data, model_params, config, device):
    if len(test_data) < config.SEQ_LENGTH:
        print(f"Test data length ({len(test_data)}) < SEQ_LENGTH ({config.SEQ_LENGTH}), cannot simulate")
        return None
    model.eval()
    dt = config.DELTA_T
    time_steps = np.arange(0, len(test_data) - config.SEQ_LENGTH, 1) * dt
    results = {
        'time': time_steps,
        **{f'real_car{i}_speeds': test_data[f'speed_{i}'].values[config.SEQ_LENGTH:] for i in range(1, 7)},
        **{f'car{i}_speeds': [] for i in range(1, 7)},
        'spacings': {f'{i}_{i+1}': [] for i in range(1, 6)},
        'real_spacings': {f'{i}_{i+1}': test_data[f'spacing_{i}_{i+1}'].values[config.SEQ_LENGTH:] for i in range(1, 6)},
        'accelerations': [],
        'D_f_list': [],
        'D_r_list': [],
        'phi_list': []
    }
    if len(results['real_car1_speeds']) == 0:
        print("No valid test data after slicing")
        return None
    results['car1_speeds'] = list(results['real_car1_speeds'])
    states = {f'car{i}_speed': test_data[f'speed_{i}'].iloc[config.SEQ_LENGTH-1] for i in range(2, 7)}
    states.update({f'spacing_{i}_{i+1}': max(0.0, test_data[f'spacing_{i}_{i+1}'].iloc[config.SEQ_LENGTH-1]) for i in range(1, 6)})  # Changed to allow 0
    states['car1_speed'] = test_data['speed_1'].iloc[config.SEQ_LENGTH-1]
    print(f"Initial spacings: spacing_1_2={states['spacing_1_2']:.4f}, spacing_2_3={states['spacing_2_3']:.4f}, "
          f"spacing_3_4={states['spacing_3_4']:.4f}, spacing_4_5={states['spacing_4_5']:.4f}, "
          f"spacing_5_6={states['spacing_5_6']:.4f}")
    idm_calcs = {f'car{i}': UnifiedIDMCalculator(model_params[f'car{i}']['idm'], device) for i in [2, 3, 5, 6]}
    test_accelerations = []
    v3_history = list(test_data['speed_3'].iloc[max(0, config.SEQ_LENGTH-10):config.SEQ_LENGTH])
    v4_history = list(test_data['speed_4'].iloc[max(0, config.SEQ_LENGTH-10):config.SEQ_LENGTH])
    with torch.no_grad():
        for i in range(len(time_steps)):
            try:
                car1_speed = test_data['speed_1'].iloc[config.SEQ_LENGTH + i]
                car2_speed = np.clip(states['car2_speed'] + idm_calcs['car2'].compute(states['car2_speed'], car1_speed, states['spacing_1_2']) * dt, config.MIN_SPEED, config.MAX_SPEED)
                car3_speed = np.clip(states['car3_speed'] + idm_calcs['car3'].compute(states['car3_speed'], car2_speed, states['spacing_2_3']) * dt, config.MIN_SPEED, config.MAX_SPEED)
                current_states = {f'speed_{j}': states[f'car{j}_speed'] for j in range(1, 7)}
                current_states.update({f'spacing_{j}_{j+1}': states[f'spacing_{j}_{j+1}'] for j in range(1, 6)})
                current_states['speed_1'] = car1_speed
                phi, D_f, D_r, _ = EnhancedFeatureProcessor.compute_dynamic_phi_with_components(current_states, config)
                results['D_f_list'].append(D_f)
                results['D_r_list'].append(D_r)
                results['phi_list'].append(phi)
                v3_history.append(car3_speed)
                v4_history.append(states['car4_speed'])
                if len(v3_history) > 10:
                    v3_history, v4_history = v3_history[-10:], v4_history[-10:]
                prev_speed_delta = 0.0 if i == 0 else states['car4_speed'] - states.get('prev_car4_speed', states['car4_speed'])
                enhanced_features = EnhancedFeatureProcessor.compute_features(
                    car3_speed, states['car4_speed'], states['spacing_3_4'], prev_speed_delta, v3_history[-10:], v4_history[-10:]
                ).unsqueeze(0).unsqueeze(0).to(device)
                dynamic_acc = model(enhanced_features, torch.tensor(phi, device=device, dtype=torch.float32), states['spacing_3_4']).item()
                acceleration = dynamic_acc
                if states['spacing_3_4'] <= config.SPACING_CRITICAL:
                    deceleration_factor = 0.2 * (1.0 - states['spacing_3_4'] / config.SPACING_CRITICAL)
                    acceleration = max(acceleration, -deceleration_factor)
                acceleration = max(acceleration, config.MIN_ACC)  # Removed max_acc limit
                phi_factor = 0.3 * (phi / (math.pi/2))  # Always active, increased to 0.3
                car4_speed = states['car4_speed'] + acceleration * dt + phi_factor * dt
                car4_speed = max(car4_speed, 0.0)
                car5_speed = np.clip(states['car5_speed'] + idm_calcs['car5'].compute(states['car5_speed'], car4_speed, states['spacing_4_5']) * dt, config.MIN_SPEED, config.MAX_SPEED)
                car6_speed = np.clip(states['car6_speed'] + idm_calcs['car6'].compute(states['car6_speed'], car5_speed, states['spacing_5_6']) * dt, config.MIN_SPEED, config.MAX_SPEED)
                speeds = [car1_speed, car2_speed, car3_speed, car4_speed, car5_speed, car6_speed]
                displacements = [speed * dt for speed in speeds]
                for j in range(1, 6):
                    states[f'spacing_{j}_{j+1}'] = max(0.0, states[f'spacing_{j}_{j+1}'] + displacements[j-1] - displacements[j])
                    results['spacings'][f'{j}_{j+1}'].append(max(0.0, states[f'spacing_{j}_{j+1}']))
                for j in range(2, 7):
                    results[f'car{j}_speeds'].append(speeds[j-1])
                results['accelerations'].append(acceleration)
                states.update({
                    'car2_speed': car2_speed,
                    'car3_speed': car3_speed,
                    'car4_speed': car4_speed,
                    'car5_speed': car5_speed,
                    'car6_speed': car6_speed
                })
            except Exception as sim_error:
                print(f"Simulation error at step {i}: {sim_error} (Full traceback: {traceback.format_exc()})")
                if i > 0 and i < len(time_steps) - 1:
                    results['accelerations'].append(results['accelerations'][-1] if results['accelerations'] else 0.0)
                    for j in range(2, 7):
                        results[f'car{j}_speeds'].append(results[f'car{j}_speeds'][-1] if results[f'car{j}_speeds'] else 0.1)
                    for key in results['spacings']:
                        last_valid_spacing = results['spacings'][key][-1] if results['spacings'][key] else 10.0
                        results['spacings'][key].append(max(0.0, last_valid_spacing))
                    results['D_f_list'].append(results['D_f_list'][-1] if results['D_f_list'] else 0.0)
                    results['D_r_list'].append(results['D_r_list'][-1] if results['D_r_list'] else 0.0)
                    results['phi_list'].append(results['phi_list'][-1] if results['phi_list'] else math.pi/4)
                else:
                    break
    print("Simulation completed")
    print("Phi stats: mean=", np.mean(results['phi_list']), "std=", np.std(results['phi_list']))
    if results['accelerations']:
        print("Acceleration range:", min(results['accelerations']), "to", max(results['accelerations']))
    else:
        print("Acceleration range: No data (list is empty)")
    return results

def save_results_to_excel(simulation_results, idm_results, results_dir):
    try:
        excel_path = os.path.join(results_dir, 'simulation_results.xlsx')
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            time_len = len(simulation_results.get('time', []))
            trajectory_data = {'Time(s)': simulation_results.get('time', [])}
            for i in range(1, 7):
                real_speeds = np.array(simulation_results.get(f'real_car{i}_speeds', np.zeros(time_len)))
                if len(real_speeds) > 1:
                    real_acc = np.diff(real_speeds) / (1/30)
                    real_acc = np.pad(real_acc, (0, 1), mode='constant', constant_values=0.0)
                else:
                    real_acc = np.zeros(time_len)
                trajectory_data[f'Real_Car{i}_Acc'] = real_acc[:time_len]
            for i in range(1, 7):
                idm_speeds = np.array(idm_results.get(f'car{i}_speeds', np.zeros(time_len)))
                if len(idm_speeds) > 1:
                    idm_acc = np.diff(idm_speeds) / (1/30)
                    idm_acc = np.pad(idm_acc, (0, 1), mode='constant', constant_values=0.0)
                else:
                    idm_acc = np.zeros(time_len)
                trajectory_data[f'IDM_Car{i}_Acc'] = idm_acc[:time_len]
            dyn_acc = np.array(simulation_results.get('accelerations', np.zeros(time_len)))
            if dyn_acc.size < time_len and dyn_acc.size > 0:
                dyn_acc = np.pad(dyn_acc, (0, time_len - dyn_acc.size), mode='constant', constant_values=0.0)
            elif dyn_acc.size == 0:
                dyn_acc = np.zeros(time_len)
            trajectory_data[f'Dynamic_Car4_Acc'] = dyn_acc[:time_len]
            for i in range(1, 7):
                real_speeds = np.array(simulation_results.get(f'real_car{i}_speeds', np.zeros(time_len)))
                trajectory_data[f'Real_Car{i}_Speed'] = real_speeds[:time_len]
            for i in range(1, 7):
                idm_speeds = np.array(idm_results.get(f'car{i}_speeds', np.zeros(time_len)))
                trajectory_data[f'IDM_Car{i}_Speed'] = idm_speeds[:time_len]
            for i in range(1, 7):
                dyn_speeds = np.array(simulation_results.get(f'car{i}_speeds', np.zeros(time_len)))
                if dyn_speeds.size < time_len and dyn_speeds.size > 0:
                    dyn_speeds = np.pad(dyn_speeds, (0, time_len - dyn_speeds.size), mode='constant', constant_values=0.0)
                elif dyn_speeds.size == 0:
                    dyn_speeds = np.zeros(time_len)
                trajectory_data[f'Dynamic_Car{i}_Speed'] = dyn_speeds[:time_len]
            for i in range(1, 6):
                real_spacing = np.array(simulation_results['real_spacings'].get(f'{i}_{i+1}', np.zeros(time_len)))
                trajectory_data[f'Real_Spacing_{i}_{i+1}'] = real_spacing[:time_len]
            for i in range(1, 6):
                idm_spacing = np.array(idm_results['spacings'].get(f'{i}_{i+1}', np.full(time_len, 10.0)))
                trajectory_data[f'IDM_Spacing_{i}_{i+1}'] = idm_spacing[:time_len]
            for i in range(1, 6):
                dyn_spacing = np.array(simulation_results['spacings'].get(f'{i}_{i+1}', np.full(time_len, 10.0)))
                if dyn_spacing.size < time_len and dyn_spacing.size > 0:
                    dyn_spacing = np.pad(dyn_spacing, (0, time_len - dyn_spacing.size), mode='constant', constant_values=dyn_spacing[-1] if dyn_spacing.size > 0 else 10.0)
                elif dyn_spacing.size == 0:
                    dyn_spacing = np.full(time_len, 10.0)
                trajectory_data[f'Dynamic_Spacing_{i}_{i+1}'] = dyn_spacing[:time_len]
            df_trajectory = pd.DataFrame(trajectory_data)
            df_trajectory.to_excel(writer, sheet_name='Trajectories', index=False)
            danger_data = {
                'Time(s)': simulation_results.get('time', [])[:time_len],
                'D_f': np.array(simulation_results.get('D_f_list', np.zeros(time_len)))[:time_len],
                'D_r': np.array(simulation_results.get('D_r_list', np.zeros(time_len)))[:time_len],
                'Phi': np.array(simulation_results.get('phi_list', np.zeros(time_len)))[:time_len]
            }
            df_danger = pd.DataFrame(danger_data)
            df_danger.to_excel(writer, sheet_name='DangerIndicators', index=False)
            stats_data = {
                'Metric': [
                    'Mean Real Acc', 'Std Real Acc', 'Mean IDM Acc', 'Std IDM Acc',
                    'Mean Dynamic Acc', 'Std Dynamic Acc', 'Mean D_f', 'Mean D_r', 'Mean Phi'
                ],
                'Value': [
                    np.mean([abs(x) for x in np.diff(simulation_results.get('real_car4_speeds', np.zeros(time_len))) / (1/30)]),
                    np.std(np.diff(simulation_results.get('real_car4_speeds', np.zeros(time_len))) / (1/30)),
                    np.mean([abs(x) for x in np.diff(idm_results.get('car4_speeds', np.zeros(time_len))) / (1/30)]),
                    np.std(np.diff(idm_results.get('car4_speeds', np.zeros(time_len))) / (1/30)),
                    np.mean([abs(x) for x in simulation_results.get('accelerations', np.zeros(time_len))]),
                    np.std(simulation_results.get('accelerations', np.zeros(time_len))),
                    np.mean(simulation_results.get('D_f_list', np.zeros(time_len))),
                    np.mean(simulation_results.get('D_r_list', np.zeros(time_len))),
                    np.mean(simulation_results.get('phi_list', np.zeros(time_len)))
                ]
            }
            df_stats = pd.DataFrame(stats_data)
            df_stats.to_excel(writer, sheet_name='Statistics', index=False)
        print(f"Excel results saved: {excel_path}")
        return excel_path
    except Exception as e:
        print(f"Error saving to Excel: {e}")
        return None

def create_enhanced_analysis_plots(simulation_results, idm_results, training_history, model_params, results_dir, config=Config()):
    plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Arial'], 'axes.unicode_minus': False, 'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 10, 'legend.fontsize': 8, 'xtick.labelsize': 9, 'ytick.labelsize': 9})
    print("Creating analysis plots...")
    results = simulation_results
    if results is None or len(results.get('time', [])) == 0:
        print("No valid simulation results or time steps")
        return
    dt, time_steps = config.DELTA_T, results['time']
    for key in ['real_car1_speeds', 'real_car2_speeds', 'real_car3_speeds', 'real_car4_speeds', 'real_car5_speeds', 'real_car6_speeds', 'car1_speeds', 'car2_speeds', 'car3_speeds', 'car4_speeds', 'car5_speeds', 'car6_speeds', 'accelerations', 'D_f_list', 'D_r_list', 'phi_list']:
        results[key] = np.array(results.get(key, []))
    for key in results.get('spacings', {}):
        results['spacings'][key] = np.array(results['spacings'].get(key, []))
    for key in results.get('real_spacings', {}):
        results['real_spacings'][key] = np.array(results['real_spacings'].get(key, []))
    for key in ['car1_speeds', 'car2_speeds', 'car3_speeds', 'car4_speeds', 'car5_speeds', 'car6_speeds', 'accelerations']:
        idm_results[key] = np.array(idm_results.get(key, []))
    if 'spacings' not in idm_results:
        idm_results['spacings'] = {f'{i}_{i+1}': np.array([]) for i in range(1, 6)}
    else:
        for key in [f'{i}_{i+1}' for i in range(1, 6)]:
            if key not in idm_results['spacings']:
                idm_results['spacings'][key] = np.array([])
            else:
                idm_results['spacings'][key] = np.array(idm_results['spacings'][key])
    car_energies, avg_speeds = {'real': {}, 'idm': {}, 'dynamic': {}}, {'real': {}, 'idm': {}, 'dynamic': {}}
    for car_num in range(1, 7):
        real_speeds = np.array(results[f'real_car{car_num}_speeds'])
        real_acc = np.diff(real_speeds) / dt if len(real_speeds) > 1 else np.zeros(len(real_speeds))
        car_energies['real'][f'car{car_num}'] = np.sum(0.5 * real_acc ** 2) * dt if len(real_acc) > 0 else 0.0
        avg_speeds['real'][f'car{car_num}'] = np.mean(real_speeds) if len(real_speeds) > 0 else 0.0
        idm_speeds = np.array(idm_results[f'car{car_num}_speeds'])
        idm_acc = np.diff(idm_speeds) / dt if len(idm_speeds) > 1 else np.zeros(len(idm_speeds))
        car_energies['idm'][f'car{car_num}'] = np.sum(0.5 * idm_acc ** 2) * dt if len(idm_acc) > 0 else 0.0
        avg_speeds['idm'][f'car{car_num}'] = np.mean(idm_speeds) if len(idm_speeds) > 0 else 0.0
        dyn_speeds = np.array(results[f'car{car_num}_speeds'])
        dyn_acc = np.array(results['accelerations']) if car_num == 4 else np.diff(dyn_speeds) / dt if len(dyn_speeds) > 1 else np.zeros(len(dyn_speeds))
        if car_num == 4 and len(dyn_acc) < len(dyn_speeds) and len(dyn_acc) > 0:
            dyn_acc = np.pad(dyn_acc, (0, len(dyn_speeds) - len(dyn_acc)), mode='constant', constant_values=0.0)
        elif car_num == 4 and len(dyn_acc) == 0:
            dyn_acc = np.zeros(len(dyn_speeds))
        car_energies['dynamic'][f'car{car_num}'] = np.sum(0.5 * dyn_acc ** 2) * dt if len(dyn_acc) > 0 else 0.0
        avg_speeds['dynamic'][f'car{car_num}'] = np.mean(dyn_speeds) if len(dyn_speeds) > 0 else 0.0
    print("Dynamic Car4 Speed range:", min(results['car4_speeds']) if len(results['car4_speeds']) > 0 else 0, "to", max(results['car4_speeds']) if len(results['car4_speeds']) > 0 else 0)
    print("Dynamic Car4 Acc range:", min(results['accelerations']) if len(results['accelerations']) > 0 else 0, "to", max(results['accelerations']) if len(results['accelerations']) > 0 else 0)
    fig = plt.figure(figsize=(25, 16))
    plt.subplots_adjust(left=0.04, right=0.96, top=0.92, bottom=0.08, hspace=0.25, wspace=0.15)
    car_colors = ['black', 'red', 'green', 'blue', 'magenta', 'cyan']
    car_labels_real = [f'Car{i} (Real)' for i in range(1, 7)]
    ax_real_acc = plt.subplot(4, 5, 1)
    for i, (color, label) in enumerate(zip(car_colors, car_labels_real)):
        speeds = np.array(results[f'real_car{i+1}_speeds'])
        accelerations = np.diff(speeds) / dt if len(speeds) > 1 else np.zeros(len(speeds))
        ax_real_acc.plot(time_steps[:len(accelerations)], accelerations, color=color, linewidth=1.0, label=label, alpha=0.8)
    ax_real_acc.set_title('Real Acceleration Trajectories', fontsize=12, fontweight='bold')
    ax_real_acc.set_ylabel('Acceleration (m/s²)', fontsize=10)
    ax_real_acc.legend(fontsize=8, loc='upper right')
    ax_real_acc.grid(True, alpha=0.3)
    ax_idm_acc = plt.subplot(4, 5, 2)
    for i in range(1, 7):
        speeds = np.array(idm_results[f'car{i}_speeds'])
        accelerations = np.diff(speeds) / dt if len(speeds) > 1 else np.zeros(len(speeds))
        label = f'Car{i} (IDM)' if i != 1 else 'Car1 (Real)'
        ax_idm_acc.plot(time_steps[:len(accelerations)], accelerations, color=car_colors[i-1], linewidth=1.0, label=label, alpha=0.8)
    ax_idm_acc.set_title('IDM Verification: Accelerations', fontsize=12, fontweight='bold')
    ax_idm_acc.set_ylabel('Acceleration (m/s²)', fontsize=10)
    ax_idm_acc.legend(fontsize=8, loc='upper right')
    ax_idm_acc.grid(True, alpha=0.3)
    ax_dyn_acc = plt.subplot(4, 5, 3)
    for j in range(1, 7):
        if j == 1:
            speeds = np.array(results['real_car1_speeds'])
            accelerations = np.diff(speeds) / dt if len(speeds) > 1 else np.zeros(len(speeds))
            label = 'Car1 (Real)'
        elif j == 4:
            accelerations = np.array(results['accelerations'])
            if len(accelerations) == 0:
                accelerations = np.zeros(len(time_steps) - 1)
            label = 'Car4 (LSTM)'
        else:
            speeds = np.array(results[f'car{j}_speeds'])
            accelerations = np.diff(speeds) / dt if len(speeds) > 1 else np.zeros(len(speeds))
            label = f'Car{j} (IDM)'
        ax_dyn_acc.plot(time_steps[:len(accelerations)], accelerations, color=car_colors[j-1], linewidth=1.0, label=label, alpha=0.8 if j != 4 else 1.0)
    ax_dyn_acc.set_title('Dynamic Phi: Accelerations (Car4 LSTM)', fontsize=12, fontweight='bold')
    ax_dyn_acc.set_ylabel('Acceleration (m/s²)', fontsize=10)
    ax_dyn_acc.legend(fontsize=8, loc='upper right')
    ax_dyn_acc.grid(True, alpha=0.3)
    ax_real_speed = plt.subplot(4, 5, 6)
    for i, (color, label) in enumerate(zip(car_colors, car_labels_real)):
        ax_real_speed.plot(time_steps[:len(results[f'real_car{i+1}_speeds'])], results[f'real_car{i+1}_speeds'], color=color, linewidth=1.0, label=label, alpha=0.8)
    ax_real_speed.set_title('Real Speed Trajectories', fontsize=12, fontweight='bold')
    ax_real_speed.set_ylabel('Speed (m/s)', fontsize=10)
    ax_real_speed.legend(fontsize=8, loc='upper right')
    ax_real_speed.grid(True, alpha=0.3)
    ax_idm_speed = plt.subplot(4, 5, 7)
    for i in range(1, 7):
        speeds = np.array(idm_results[f'car{i}_speeds'])
        label = f'Car{i} (IDM)' if i != 1 else 'Car1 (Real)'
        ax_idm_speed.plot(time_steps[:len(speeds)], speeds, color=car_colors[i-1], linewidth=1.0, label=label, alpha=0.8)
    ax_idm_speed.set_title('IDM Verification: Speeds', fontsize=12, fontweight='bold')
    ax_idm_speed.set_ylabel('Speed (m/s)', fontsize=10)
    ax_idm_speed.legend(fontsize=8, loc='upper right')
    ax_idm_speed.grid(True, alpha=0.3)
    ax_dyn_speed = plt.subplot(4, 5, 8)
    for j in range(1, 7):
        if j == 1:
            speeds = np.array(results['real_car1_speeds'])
            label = 'Car1 (Real)'
        else:
            speeds = np.array(results[f'car{j}_speeds'])
            label = f'Car{j} (LSTM)' if j == 4 else f'Car{j} (IDM)'
        ax_dyn_speed.plot(time_steps[:len(speeds)], speeds, color=car_colors[j-1], linewidth=1.0, linestyle='-' if j <= 3 or j >= 5 else '--', label=label, alpha=0.8)
    ax_dyn_speed.set_title('Dynamic Phi: Speeds (Car4 LSTM)', fontsize=12, fontweight='bold')
    ax_dyn_speed.set_ylabel('Speed (m/s)', fontsize=10)
    ax_dyn_speed.legend(fontsize=8, loc='upper right')
    ax_dyn_speed.grid(True, alpha=0.3)
    spacing_colors = ['blue', 'green', 'red', 'magenta', 'cyan']
    spacing_labels_real = [f'Car {i}-{i+1} (Real)' for i in range(1, 6)]
    ax_real_spacing = plt.subplot(4, 5, 11)
    for key, s_color, label in zip(['1_2', '2_3', '3_4', '4_5', '5_6'], spacing_colors, spacing_labels_real):
        ax_real_spacing.plot(time_steps[:len(results['real_spacings'][key])], results['real_spacings'][key], color=s_color, linewidth=1.0, label=label, alpha=0.8)
    ax_real_spacing.set_title('Real Spacing Trajectories', fontsize=12, fontweight='bold')
    ax_real_spacing.set_ylabel('Spacing (m)', fontsize=10)
    ax_real_spacing.legend(fontsize=8, loc='upper right')
    ax_real_spacing.grid(True, alpha=0.3)
    spacing_labels_idm = [f'Car {i}-{i+1} (IDM)' for i in range(1, 6)]
    ax_idm_spacing = plt.subplot(4, 5, 12)
    for key, s_color, label in zip(['1_2', '2_3', '3_4', '4_5', '5_6'], spacing_colors, spacing_labels_idm):
        spacing_data = idm_results['spacings'].get(key, [])
        if len(spacing_data) > 0:
            ax_idm_spacing.plot(time_steps[:len(spacing_data)], spacing_data, color=s_color, linewidth=1.0, label=label, alpha=0.8)
    ax_idm_spacing.set_title('IDM Verification: Spacings', fontsize=12, fontweight='bold')
    ax_idm_spacing.set_ylabel('Spacing (m)', fontsize=10)
    ax_idm_spacing.legend(fontsize=8, loc='upper right')
    ax_idm_spacing.grid(True, alpha=0.3)
    spacing_labels_sim = [f'Car {i}-{i+1} (Sim)' for i in range(1, 6)]
    ax_spacing = plt.subplot(4, 5, 13)
    for key, s_color, label in zip(['1_2', '2_3', '3_4', '4_5', '5_6'], spacing_colors, spacing_labels_sim):
        spacing_data = results['spacings'].get(key, [])
        if len(spacing_data) > 0:
            ax_spacing.plot(time_steps[:len(spacing_data)], spacing_data, color=s_color, linewidth=1.0, label=label, alpha=0.8)
    ax_spacing.set_title('Dynamic Phi: Spacings (Car4 LSTM)', fontsize=12, fontweight='bold')
    ax_spacing.set_ylabel('Spacing (m)', fontsize=10)
    ax_spacing.legend(fontsize=8, loc='upper right')
    ax_spacing.grid(True, alpha=0.3)
    ax_phi = plt.subplot(4, 5, 16)
    ax_phi.plot(time_steps[:len(results['phi_list'])], results['phi_list'], color='purple', linewidth=1.5, label='Phi')
    ax_phi.plot(time_steps[:len(results['D_f_list'])], results['D_f_list'], color='blue', linewidth=1.5, label='D_f (Front Danger)')
    ax_phi.plot(time_steps[:len(results['D_r_list'])], results['D_r_list'], color='red', linewidth=1.5, label='D_r (Rear Danger)')
    ax_phi.set_title('Phi, D_f, D_r Trajectories', fontsize=12, fontweight='bold')
    ax_phi.set_ylabel('Value (radians for Phi, unitless for D_f, D_r)', fontsize=10)
    ax_phi.set_xlabel('Time (s)', fontsize=10)
    ax_phi.legend(fontsize=8)
    ax_phi.grid(True, alpha=0.3)
    ax_energy = plt.subplot(4, 5, 18)
    car_names = [f'Car{i}' for i in range(1, 7)]
    x_pos = np.arange(len(car_names))
    bar_width = 0.25
    energy_colors = ['#4ecdc4', '#ff6b6b', '#96ceb4']
    real_energies = [car_energies['real'][f'car{i}'] for i in range(1, 7)]
    ax_energy.bar(x_pos - bar_width, real_energies, bar_width, label='Real', color=energy_colors[0], alpha=0.8)
    idm_energies = [car_energies['idm'][f'car{i}'] for i in range(1, 7)]
    ax_energy.bar(x_pos, idm_energies, bar_width, label='IDM', color=energy_colors[1], alpha=0.8)
    dynamic_energies = [car_energies['dynamic'][f'car{i}'] for i in range(1, 7)]
    ax_energy.bar(x_pos + bar_width, dynamic_energies, bar_width, label='Dynamic', color=energy_colors[2], alpha=0.8)
    for i, (real_e, idm_e, dyn_e) in enumerate(zip(real_energies, idm_energies, dynamic_energies)):
        ax_energy.annotate(f'{real_e:.2f}', xy=(x_pos[i] - bar_width, real_e), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=7)
        ax_energy.annotate(f'{idm_e:.2f}', xy=(x_pos[i], idm_e), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=7)
        ax_energy.annotate(f'{dyn_e:.2f}', xy=(x_pos[i] + bar_width, dyn_e), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=7)
    ax_energy.set_title('Energy Consumption by Scenario', fontsize=12, fontweight='bold')
    ax_energy.set_xlabel('Vehicles', fontsize=10)
    ax_energy.set_ylabel('Energy ((m/s²)²·s)', fontsize=10)
    ax_energy.set_xticks(x_pos)
    ax_energy.set_xticklabels(car_names, fontsize=9)
    ax_energy.legend(fontsize=7)
    ax_energy.grid(True, alpha=0.3, axis='y')
    ax_avg_speed = plt.subplot(4, 5, 19)
    real_speeds = [avg_speeds['real'][f'car{i}'] for i in range(1, 7)]
    ax_avg_speed.bar(x_pos - bar_width, real_speeds, bar_width, label='Real', color=energy_colors[0], alpha=0.8)
    idm_speeds = [avg_speeds['idm'][f'car{i}'] for i in range(1, 7)]
    ax_avg_speed.bar(x_pos, idm_speeds, bar_width, label='IDM', color=energy_colors[1], alpha=0.8)
    dynamic_speeds = [avg_speeds['dynamic'][f'car{i}'] for i in range(1, 7)]
    ax_avg_speed.bar(x_pos + bar_width, dynamic_speeds, bar_width, label='Dynamic', color=energy_colors[2], alpha=0.8)
    for i, (real_s, idm_s, dyn_s) in enumerate(zip(real_speeds, idm_speeds, dynamic_speeds)):
        ax_avg_speed.annotate(f'{real_s:.2f}', xy=(x_pos[i] - bar_width, real_s), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=7)
        ax_avg_speed.annotate(f'{idm_s:.2f}', xy=(x_pos[i], idm_s), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=7)
        ax_avg_speed.annotate(f'{dyn_s:.2f}', xy=(x_pos[i] + bar_width, dyn_s), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=7)
    ax_avg_speed.set_title('Average Speeds by Scenario', fontsize=12, fontweight='bold')
    ax_avg_speed.set_xlabel('Vehicles', fontsize=10)
    ax_avg_speed.set_ylabel('Speed (m/s)', fontsize=10)
    ax_avg_speed.set_xticks(x_pos)
    ax_avg_speed.set_xticklabels(car_names, fontsize=9)
    ax_avg_speed.legend(fontsize=7)
    ax_avg_speed.grid(True, alpha=0.3, axis='y')
    plt.suptitle('Dynamic SVO Analysis Results (Improved)', fontsize=16, fontweight='bold', y=0.995)
    save_path = os.path.join(results_dir, 'dynamic_svo_analysis.png')
    plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Dynamic SVO analysis plot saved: {save_path}")
    fig_training = plt.figure(figsize=(8, 6))
    ax_training = fig_training.add_subplot(1, 1, 1)
    if training_history:
        losses = [x for x in training_history if 0.001 <= x <= 1000]
        if losses:
            epochs = range(1, len(losses) + 1)
            ax_training.plot(epochs, losses, label='Training Loss', linewidth=2.0, marker='o', markersize=4, color='purple', alpha=0.8)
            ax_training.fill_between(epochs, losses, alpha=0.2, color='purple')
    ax_training.set_title('Training Loss Over Epochs', fontsize=12, fontweight='bold')
    ax_training.set_xlabel('Epoch', fontsize=10)
    ax_training.set_ylabel('Loss', fontsize=10)
    ax_training.legend(fontsize=9)
    ax_training.grid(True, alpha=0.3)
    training_save_path = os.path.join(results_dir, 'training_loss.png')
    plt.savefig(training_save_path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Training loss plot saved: {training_save_path}")

def main():
    config = Config()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Set random seeds for reproducibility
    seed = 42
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Dynamic SVO Training\nDevice: {device} | Epochs: {config.NUM_EPOCHS} | Max Rows: {config.MAX_ROWS} | Seed: {seed}\n{'=' * 60}")
    try:
        data, _ = load_and_validate_data(max_rows=config.MAX_ROWS, config=config)
        if data is None:
            print("Failed to load data")
            return None
        model_params = {
            "car2": {"idm": {"v_0": 10.0, "a_max": 1.95, "b": 1.51, "s_0": 6.59, "T": 0.74, "delta": 5.21}},
            "car3": {"idm": {"v_0": 5.36, "a_max": 1.61, "b": 0.25, "s_0": 4.90, "T": 1.96, "delta": 2.00}},
            "car4": {"idm": {"v_0": 5.36, "a_max": 1.61, "b": 0.25, "s_0": 4.90, "T": 1.96, "delta": 2.00}},
            "car5": {"idm": {"v_0": 4.85, "a_max": 3.00, "b": 0.74, "s_0": 5.73, "T": 0.50, "delta": 5.16}},
            "car6": {"idm": {"v_0": 5.15, "a_max": 0.62, "b": 0.46, "s_0": 6.70, "T": 0.50, "delta": 3.03}}
        }
        results_dir = setup_validation(config)
        print(f"Results directory: {results_dir}")
        split_idx = int(len(data) * 0.8)
        train_data, test_data = data.iloc[:split_idx], data.iloc[split_idx:]
        print(f"Training: {len(train_data)} rows | Testing: {len(test_data)} rows")
        print("Starting training...")
        model, loss_fn, history = train_enhanced(train_data, model_params, config, device, results_dir)
        print("Running simulation...")
        simulation_results = simulate_enhanced(model, test_data, model_params, config, device)
        if simulation_results is None:
            print("Simulation failed")
            return None
        print("Computing IDM verification...")
        idm_results = compute_idm_verification(simulation_results, model_params, config, compute_spacings=True)
        print("Creating analysis plots...")
        create_enhanced_analysis_plots(simulation_results, idm_results, history, model_params, results_dir, config)
        print("Saving results to Excel...")
        save_results_to_excel(simulation_results, idm_results, results_dir)
        print(f"\n{'=' * 60}")
        print(f"All results saved to: {results_dir}")
        print(f"{'=' * 60}")
        return model, history
    except Exception as e:
        print(f"Error: {e}")
        traceback.print_exc()
        return None

if __name__ == "__main__":
    main()